# Step 3: Reconciliation

**Why an investor cares.** If the data cube does not tie to the management accounts, nothing built on it can be trusted, and the seller's adviser will attack every number that comes out of it. Reconciliation earns the right to do the analysis. It also tests the management accounts themselves: where the invoices and the reported figures disagree, one of them is wrong, and the investor is about to pay a multiple of the reported figure.

This notebook reads the `recon_*` tables built by `sql/04_reconciliation.sql` and saves them to `outputs/tables/` for the databook. The P06 anomaly treatment is still at its Step 2 default (`as_reported`) until it is settled below.

In [1]:
import pandas as pd
from nb_utils import connect, q, md_table, gbp, pct, TABLES

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)
con = connect()
print("current anomaly treatment:", q(con, "SELECT anomaly_treatment FROM cfg_settings").iloc[0, 0])
q(con, "SELECT * FROM cfg_materiality")

current anomaly treatment: as_reported


,flag_gbp,material_pct,annual_pct
0,1000.0,0.005,0.001


## 1. Anomaly options

Cube net revenue under each P06 treatment, against the management accounts' reported `total_revenue` and against the sum of their four product-line columns.

In [2]:
opt_sum = q(con, "SELECT * FROM recon_options_summary")
opt_sum.to_csv(TABLES / "step3_anomaly_options.csv", index=False)
opt_sum

,treatment,months_tied_to_total,months_tied_to_lines,abs_diff_vs_total,abs_diff_vs_lines,net_vs_total_2022,net_vs_total_2023,net_vs_total_2024,net_vs_total_2025
0,as_reported,33,36,68843.82,13843.82,-5788.75,16381.04,-6573.48,-16862.63
1,flipped,33,36,65234.88,13843.82,5788.75,20618.96,1573.48,-13137.37
2,excluded,44,48,55000.00,0.00,0.00,18500.00,-2500.00,-15000.00


## 2. Monthly and annual revenue reconciliation (current setting)

In [3]:
monthly = q(con, "SELECT * FROM recon_monthly")
annual = q(con, "SELECT * FROM recon_annual")
monthly.to_csv(TABLES / "step3_recon_monthly.csv", index=False)
annual.to_csv(TABLES / "step3_recon_annual.csv", index=False)
display(annual)
monthly[monthly.status != "tied"]

,year,mgmt_total_revenue,cube_net,difference,difference_pct,material,months_tied,months_flagged
0,2022,28232151.64,28226362.89,-5788.75,-0.021,False,8,3
1,2023,32174369.95,32190750.99,16381.04,0.051,False,9,2
2,2024,37261398.51,37254825.03,-6573.48,-0.018,False,7,3
3,2025,45771419.35,45754556.72,-16862.63,-0.037,False,9,2


,month,mgmt_total_revenue,cube_net,difference,difference_pct,flagged,material,status
2,2022-03-01,2249417.71,2248715.24,-702.47,-0.031,False,False,difference
3,2022-04-01,2327025.28,2325889.20,-1136.08,-0.049,True,False,difference
6,2022-07-01,2392021.50,2390990.52,-1030.98,-0.043,True,False,difference
8,2022-09-01,2404716.28,2401797.06,-2919.22,-0.121,True,False,difference
12,2023-01-01,2553770.31,2553220.31,-550.00,-0.022,False,False,difference
17,2023-06-01,2716929.59,2735429.59,18500.00,0.681,True,True,difference
18,2023-07-01,2681733.32,2680164.36,-1568.96,-0.059,True,False,difference
26,2024-03-01,3051622.66,3037818.19,-13804.47,-0.452,True,False,difference
27,2024-04-01,3083644.32,3082589.50,-1054.82,-0.034,True,False,difference
28,2024-05-01,3078701.34,3078008.96,-692.38,-0.022,False,False,difference


## 3. Product-line reconciliation (current setting)

In [4]:
pl_annual = q(con, "SELECT * FROM recon_product_line_annual")
pl_monthly = q(con, "SELECT * FROM recon_product_line_monthly")
pl_annual.to_csv(TABLES / "step3_recon_product_line_annual.csv", index=False)
pl_monthly.to_csv(TABLES / "step3_recon_product_line_monthly.csv", index=False)
pl_annual.pivot(index="product_line", columns="year", values="difference")

year,2022,2023,2024,2025
product_line,,,,
Analytics Add-on,0.00,-550.00,-1072.63,-599.63
Core Platform,-5086.28,-1568.96,-1682.12,-1263.00
Implementation Services,0.00,0.00,0.00,0.00
Payments Module,-702.47,0.00,-1318.73,0.00


## 4. Management accounts: internal consistency

In [5]:
internal = q(con, "SELECT * FROM recon_mgmt_internal")
internal.to_csv(TABLES / "step3_mgmt_internal.csv", index=False)
internal[internal.lines_total_mismatch_flag | (internal.gp_arithmetic_diff.abs() > 0.005) | (internal.margin_pct_diff.abs() > 0.00005)]

,month,revenue_sum_of_lines,total_revenue,lines_less_total,gp_arithmetic_diff,margin_pct_diff,lines_total_mismatch_flag
17,2023-06-01,2735429.59,2716929.59,18500.0,0.0,0.0,True
26,2024-03-01,3039622.66,3051622.66,-12000.0,0.0,0.0,True
34,2024-11-01,3200376.09,3190876.09,9500.0,0.0,0.0,True
37,2025-02-01,3787073.54,3802073.54,-15000.0,0.0,0.0,True


## 5. Costs against management cost of sales

In [6]:
costs = q(con, "SELECT * FROM recon_costs_monthly")
costs.to_csv(TABLES / "step3_recon_costs.csv", index=False)
costs_y = costs.assign(year=costs.month.dt.year).groupby("year")[["cost_of_delivery", "cost_of_sales", "difference"]].sum()
print("months with a cost difference:", int((costs.difference.abs() > 0.005).sum()))
costs_y

months with a cost difference: 0


,cost_of_delivery,cost_of_sales,difference
year,,,
2022,6367072.32,6367072.32,0.0
2023,7276948.58,7276948.58,0.0
2024,8276093.15,8276093.15,0.0
2025,12008133.71,12008133.71,0.0


## 6. Source-to-reported walk (current setting)

Each line is a formula of the lines above it: cube net = raw gross billings + less duplicates + anomaly treatment + credit notes; management total = cube net + unreconciled difference.

In [7]:
walk = q(con, "SELECT * FROM recon_walk")
walk["check_cube"] = (walk.raw_gross_billings + walk.less_duplicates + walk.anomaly_treatment + walk.credit_notes - walk.cube_net).abs() < 0.005
walk["check_mgmt"] = (walk.cube_net + walk.unreconciled_difference - walk.mgmt_total).abs() < 0.005
walk.to_csv(TABLES / "step3_walk.csv", index=False)
walk

,year,raw_gross_billings,less_duplicates,anomaly_treatment,credit_notes,cube_net,unreconciled_difference,mgmt_total,check_cube,check_mgmt
0,2022,28564164.39,-8596.92,0.0,-329204.58,28226362.89,5788.75,28232151.64,True,True
1,2023,32537962.92,-70158.24,0.0,-277053.69,32190750.99,-16381.04,32174369.95,True,True
2,2024,37583315.05,-7115.70,0.0,-321374.32,37254825.03,6573.48,37261398.51,True,True
3,2025,46174479.60,-11086.40,0.0,-408836.48,45754556.72,16862.63,45771419.35,True,True


## 7. Flagged months: drill-down

Every month with an absolute difference above the flag threshold, with what the data holds in that month. `residual_after_anomaly_and_line_gap` is the difference left after removing the anomaly amount carried in the cube and the management accounts' own line-sum gap.

In [8]:
drill = q(con, "SELECT * FROM recon_drilldown")
drill.to_csv(TABLES / "step3_drilldown.csv", index=False)
drill

,month,difference,anomaly_amount_in_month,anomaly_rows_in_month,duplicates_removed_in_month,credit_notes_in_month,mgmt_lines_less_total,product_line_differences,residual_after_anomaly_and_line_gap
0,2022-04-01,-1136.08,-1136.08,1,1127.64,-14221.28,0.0,Core Platform -1136.08,0.0
1,2022-07-01,-1030.98,-1030.98,1,0.00,-27265.42,0.0,Core Platform -1030.98,0.0
2,2022-09-01,-2919.22,-2919.22,1,0.00,-16045.30,0.0,Core Platform -2919.22,0.0
3,2023-06-01,18500.00,0.00,0,460.62,-16855.72,18500.0,NaN,0.0
4,2023-07-01,-1568.96,-1568.96,2,1710.50,-13358.58,0.0,Core Platform -1568.96,0.0
5,2024-03-01,-13804.47,-1804.47,2,524.35,-26943.15,-12000.0,Core Platform -1178.12; Payments Module -626.35,0.0
6,2024-04-01,-1054.82,-1054.82,2,0.00,-31629.51,0.0,Analytics Add-on -550.82; Core Platform -504.00,0.0
7,2024-11-01,9500.00,0.00,0,0.00,-35951.25,9500.0,NaN,0.0
8,2025-02-01,-15000.00,0.00,0,2325.73,-31394.01,-15000.0,NaN,0.0
9,2025-08-01,-1263.00,-1263.00,1,0.00,-14452.78,0.0,Core Platform -1263.00,0.0
